# LangChain Retrieval Agents

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-retrieval-agents-202503.ipynb
</div>

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>

## Environment Setup

<br>

> 💡 **Important:**
>
> This notebook requires **LangChain < 0.3**.
>
> Below, you will find two options for installing the required dependencies. Choose the one that best matches your environment::
>
> 🖥️ **Running locally?**  
> → Use **Option A** to create a dedicated virtual environment (recommended).
>
> ☁️ **Using Google Colab or want a quick setup?**  
> → Use **Option B** to install the required dependencies directly.
>

<br><br>


### Option A: Use a virtual environment

Open a terminal and run the following commands.


<br>

**1. Create a virtual environment:**

```bash
python -m venv .venv/langchain-v0.2.x
```

<br>

**2. Activate the virtual environment:**

- **macOS / Linux:**
```bash
    source .venv/langchain-v0.2.x/bin/activate
```

- **Windows (PowerShell):**
```powershell
    .\.venv\langchain-v0.2.x\Scripts\Activate.ps1
```

<br>

**3. Install the required packages:**

```bash
python -m pip install \
    "langchain<0.3" \
    "langchain-core<0.3" \
    "langchain-community<0.3" \
    "langchain-openai<0.2" \
    ipykernel
```

<br>

**4. Register the environment as a Jupyter kernel:**

```bash
python -m ipykernel install \
    --user \
    --name langchain-v0.2.x \
    --display-name "Python (LangChain 0.2.x)"
```

<br>

**5. Select the correct kernel:**

Once you've completed the previous steps, do the following:
1. Open this notebook in your favourite environment (e.g., Jupyter or VS Code)
2. Select the Kernel you've just created
    - **Jupyter**: Kernel → Change Kernel → Python (LangChain 0.2.x).
    - **VS Code**: Click the Kernel selector in the top-right corner of the notebook editor → Jupyter Kernel → Python (LangChain 0.2.x)
        - Notice that you need to select "Jupyter Kernel" (not "Python Environments")
        - If "Python (LangChain 0.2.x)" doesn't appear in the kernel list, reload VS Code:
            - Press Cmd + Shift + P to open the Command Palette
            - Type Developer: Reload Window and press Enter
            - Try selecting the kernel again
3. Run the notebook as usual.

<br>

> **Notes:**
>
> - Make sure to add the directory `.venv` to your `.gitignore`
> - The virtual environment setup only needs to be completed once. The environment can then be reused for other notebooks that require **LangChain < 0.3**, without affecting your default Python environment or newer LangChain installations.

<br>

### Option B: Install dependencies directly

If you are using Google Colab, or you're having problems configuring a virtual environment, create a code cell and run the command below:

```python
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"
```

<br>


</div>

Conversational agents can struggle with data freshness, knowledge about specific domains, or accessing internal documentation. By coupling agents with retrieval augmentation tools we no longer have these problems.

One the other side, using "naive" retrieval augmentation without the use of an agent means we will retrieve contexts with *every* query. Again, this isn't always ideal as not every query requires access to external knowledge.

Merging these methods gives us the best of both worlds. In this notebook we'll learn how to do this.

To begin, we must install the prerequisite libraries that we will be using in this notebook.

## Install other dependencies we'll use in this notebook

In [1]:
%pip install datasets pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -qU "langchain-pinecone<0.2" "pinecone-notebooks==0.1.1"


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


If you're using a Jupyter notebook or a similar environment, restart the kernel to ensure the changes take effect.

## Building the Knowledge Base

We start by constructing our knowledge base. We'll use a mostly prepared dataset called **S**tanford **Qu**estion-**A**nswering **D**ataset (SQuAD) hosted on Hugging Face *Datasets*. We download it like so:

In [3]:
from datasets import load_dataset

data = load_dataset('squad', split='train')
data

/Users/luis/Desktop/ironhack_june26/1_ai_eng_lectures/.venv/langchain-v0.2.x/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using the latest cached version of the dataset since squad couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /Users/luis/.cache/huggingface/datasets/squad/plain_text/0.0.0/7b6d24c440a36b6815f21b70d25016731768db1f (last modified on Sun Aug 10 10:11:49 2025).


Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})

The dataset does contain duplicate contexts, which we can remove like so:

In [4]:
data = data.to_pandas()
data.head()

,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


In [5]:
data.drop_duplicates(subset='context', keep='first', inplace=True)

# ℹ️ Note:
# 
# Indexing the entire SQuAD dataset requires a large number of embedding
# and indexing requests, which can take a while and consume API usage. 
# 
# For this demo, we only need a small sample of contexts to showcase 
# retrieval-augmented agents, so we limit the number of rows here. 
# Increase the value "n" if you want to experiment with a larger knowledge base.
# 
data = data.sample(n=500, random_state=42).reset_index(drop=True)

data.head()

,id,title,context,question,answers
0,5727a2d2ff5b5019007d9180,Political_party,"More commonly, in cases where there are three ...",In which case are parties not likely to gain p...,{'text': ['in cases where there are three or m...
1,570e1d420b85d914000d7cd7,Antarctica,There has been some concern over the potential...,The entry of whom has caused some worry about ...,"{'text': ['visitors'], 'answer_start': [115]}"
2,572b4fecbe1ee31400cb831b,Guam,"After World War II, the Guam Organic Act of 19...",What established Guam as an unincorporated ter...,"{'text': ['Guam Organic Act of 1950'], 'answer..."
3,57261d85ec44d21400f3d8fb,Hellenistic_period,Hellenistic art saw a turn from the idealistic...,Emotion is called what in Hellenistic art?,"{'text': ['pathos'], 'answer_start': [170]}"
4,572b50a1111d821400f38e54,Guam,Guam lies between 13.2°N and 13.7°N and betwee...,How many square miles is Guam?,"{'text': ['212'], 'answer_start': [88]}"


### Initialize the Embedding Model and Vector DB

We'll be using OpenAI's `text-embedding-ada-002` model initialize via LangChain and the Pinecone vector DB. We start by initializing the embedding model, for this we need an [OpenAI API key](https://platform.openai.com/).

*(Note that OpenAI is a paid service and so running the remainder of this notebook may incur some small cost)*

In [6]:
import os
from getpass import getpass
from langchain_openai import OpenAIEmbeddings

# get API key from top-right dropdown on OpenAI website
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")

model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=OPENAI_API_KEY
)

Now we create our vector DB to store our vectors. For this we need to get a [free Pinecone API key](https://app.pinecone.io) — the API key can be found in the "API Keys" button found in the left navbar of the Pinecone dashboard.

In [7]:
from pinecone import Pinecone

# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.getenv("PINECONE_API_KEY") or getpass("Enter your Pinecone API key: ")

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [8]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

Creating an index, we set `dimension` equal to to dimensionality of Ada-002 (`1536`), and use a `metric` also compatible with Ada-002 (this can be either `cosine` or `dotproduct`). We also pass our `spec` to index initialization.

In [9]:
import time

index_name = "langchain-retrieval-agent"
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1536,  # dimensionality of ada 002
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

We should see that the new Pinecone index has a `total_vector_count` of `0`, as we haven't added any vectors yet.

## Indexing

We can perform the indexing task using the LangChain vector store object. But for now it is much faster to do it via the Pinecone python client directly. We will do this in batches of `100` or more.


> ℹ️ Why We Skip Chunking
> 
> For this demo, we won't perform any chunking. Instead, we'll index each SQuAD context paragraph as a single embedding (vector). This works well because SQuAD contexts are already short, self-contained passages.
> 




In [10]:
#
# note: 
# 
# Depending on the number of samples, running this cell may take some time 
# because it generates embeddings for each item, and 
# uploads them to the vector index.
#

from tqdm.auto import tqdm

batch_size = 100

texts = []
metadatas = []

for i in tqdm(range(0, len(data), batch_size)):
    # get end of batch
    i_end = min(len(data), i+batch_size)
    batch = data.iloc[i:i_end]
    # first get metadata fields for this record
    metadatas = [{
        'title': record['title'],
        'text': record['context']
    } for j, record in batch.iterrows()]
    # get the list of contexts / documents
    documents = batch['context']
    # create document embeddings
    embeds = embed.embed_documents(documents)
    # get IDs
    ids = batch['id']
    # add everything to pinecone
    index.upsert(vectors=zip(ids, embeds, metadatas))

100%|██████████| 5/5 [00:08<00:00,  1.79s/it]


We've indexed everything, now we can check the number of vectors in our index like so:

In [11]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 500}},
 'total_vector_count': 500}

## Creating a Vector Store and Querying

Now that we've build our index we can switch back over to LangChain. We start by initializing a vector store using the same index we just built. We do that like so:

In [12]:
from langchain.vectorstores import Pinecone

text_field = "text"  # the metadata field that contains our text

# initialize the vector store object
vectorstore = Pinecone(
    index, embed.embed_query, text_field
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_89332/3239260043.py:6: LangChainDeprecationWarning: The class `Pinecone` was deprecated in LangChain 0.0.18 and will be removed in 1.0. An updated version of the class exists in the langchain-pinecone package and should be used instead. To use it run `pip install -U langchain-pinecone` and import as `from langchain_pinecone import Pinecone`.
  vectorstore = Pinecone(
/Users/luis/Desktop/ironhack_june26/1_ai_eng_lectures/.venv/langchain-v0.2.x/lib/python3.11/site-packages/langchain_community/vectorstores/pinecone.py:68: UserWarning: Passing in `embedding` as a Callable is deprecated. Please pass in an Embeddings object instead.
  warnings.warn(


As in previous examples, we can use the `similarity_search` method to do a pure semantic search (without the generation component).

In [13]:
query = "when was the college of engineering in the University of Notre Dame established?"

vectorstore.similarity_search(
    query,  # our search query
    k=3  # return 3 most relevant docs
)

[Document(metadata={'title': 'University_of_Notre_Dame'}, page_content='The School of Architecture was established in 1899, although degrees in architecture were first awarded by the university in 1898. Today the school, housed in Bond Hall, offers a five-year undergraduate program leading to the Bachelor of Architecture degree. All undergraduate students study the third year of the program in Rome. The university is globally recognized for its Notre Dame School of Architecture, a faculty that teaches (pre-modernist) traditional and classical architecture and urban planning (e.g. following the principles of New Urbanism and New Classical Architecture). It also awards the renowned annual Driehaus Architecture Prize.'),
 Document(metadata={'title': 'Institute_of_technology'}, page_content='During the 1970s to early 1990s, the term was used to describe state owned and funded technical schools that offered both vocational and higher education. They were part of the College of Advanced Educ

Looks like we're getting good results. Let's take a look at how we can begin integrating this into a conversational agent.

## Initializing the Conversational Agent

Our conversational agent needs a Chat LLM, conversational memory, and a `RetrievalQA` chain to initialize. We create these using:

In [14]:
from langchain_openai import ChatOpenAI
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.chains import RetrievalQA

# chat completion llm
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name='gpt-3.5-turbo',
    temperature=0.0
)
# conversational memory
conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=5,
    return_messages=True
)
# retrieval qa chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

Using these we can generate an answer using the `run` method:

In [15]:
qa.run(query)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_89332/2828950282.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  qa.run(query)


'The College of Engineering at the University of Notre Dame was established in 1920.'

But this isn't yet ready for our conversational agent. For that we need to convert this retrieval chain into a tool. We do that like so:

In [16]:
from langchain.agents import Tool

tools = [
    Tool(
        name='Knowledge Base',
        func=qa.run,
        description=(
            'use this tool when answering general knowledge queries to get '
            'more information about the topic'
        )
    )
]

Now we're ready to initialize the agent itself. We'll use LangChain's `initialize_agent` helper, which wires together the LLM, tools, memory, and an agent "type" (the strategy the agent uses to reason and decide which tool to call) into a ready-to-use `AgentExecutor`.

Here we use the `chat-conversational-react-description` agent type, which is designed for chat models and uses the **ReAct** (Reason + Act) pattern: at each step the agent reasons about what to do, decides whether to call a tool (like our `Knowledge Base` tool) or answer directly, observes the result, and repeats until it can give a final answer.

![](../_images/react-pattern.png)


Key arguments:
- `tools` / `llm`: the tools and chat model the agent can use.
- `memory`: lets the agent remember previous turns in the conversation.
- `max_iterations`: caps how many reasoning/tool-call steps the agent can take before stopping (avoids infinite loops).
- `early_stopping_method='generate'`: if the agent hits the iteration limit, it will still generate a best-effort final answer instead of failing.

> ⚠️ **Note:** `initialize_agent` is deprecated in newer LangChain versions in favor of constructors like `create_react_agent`. We use it here because this notebook targets LangChain < 0.3, but keep in mind you'll see the newer approach in more recent LangChain code.

In [17]:
from langchain.agents import initialize_agent

agent = initialize_agent(
    agent='chat-conversational-react-description',
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=3,
    early_stopping_method='generate',
    memory=conversational_memory
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_89332/640423283.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


With that our retrieval augmented conversational agent is ready and we can begin using it.

### Using the Conversational Agent

To make queries we simply call the `agent` directly.

In [18]:
agent(query)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_89332/4024130983.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  agent(query)
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


```json
{
    "action": "Knowledge Base",
    "action_input": "Establishment date of the College of Engineering at the University of Notre Dame"
}
```
Observation: I don't have the specific establishment date of the College of Engineering at the University of Notre Dame based on the provided context.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The specific establishment date of the College of Engineering at the University of Notre Dame is not available."
}
```

> Finished chain.


{'input': 'when was the college of engineering in the University of Notre Dame established?',
 'chat_history': [],
 'output': 'The specific establishment date of the College of Engineering at the University of Notre Dame is not available.'}

<br>

In this case, the agent identified that it can use the 'Knowledge Base' tool ✅

<br>

Now what if we ask it a non-general knowledge question?

In [19]:
agent("what is 2 * 7?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


```json
{
    "action": "Final Answer",
    "action_input": "The result of 2 * 7 is 14."
}
```

> Finished chain.


{'input': 'what is 2 * 7?',
 'chat_history': [HumanMessage(content='when was the college of engineering in the University of Notre Dame established?'),
  AIMessage(content='The specific establishment date of the College of Engineering at the University of Notre Dame is not available.')],
 'output': 'The result of 2 * 7 is 14.'}

<br>

Perfect, the agent is able to recognize that it doesn't need to refer to it's general knowledge tool for that question ✅


<br>

Let's try some more questions.

In [20]:
agent("can you tell me some facts about the University of Notre Dame?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


```json
{
    "action": "Knowledge Base",
    "action_input": "University of Notre Dame facts"
}
```
Observation: The University of Notre Dame was established in 1842 by Father Edward Sorin, and the School of Architecture was established in 1899. The university offers a five-year undergraduate program leading to the Bachelor of Architecture degree, with all undergraduate students studying the third year of the program in Rome. The university is known for its Notre Dame School of Architecture, which focuses on traditional and classical architecture and urban planning. Additionally, the university has various undergraduate traditions like Painting The Rock, Dance Marathon, Primal Scream, and Armadillo Day.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The University of Notre Dame was established in 1842 by Father Edward Sorin. The School of Architecture was established in 1899, offering a five-year undergraduate program leading to the Bachelor of Architecture degre

{'input': 'can you tell me some facts about the University of Notre Dame?',
 'chat_history': [HumanMessage(content='when was the college of engineering in the University of Notre Dame established?'),
  AIMessage(content='The specific establishment date of the College of Engineering at the University of Notre Dame is not available.'),
  HumanMessage(content='what is 2 * 7?'),
  AIMessage(content='The result of 2 * 7 is 14.')],
 'output': 'The University of Notre Dame was established in 1842 by Father Edward Sorin. The School of Architecture was established in 1899, offering a five-year undergraduate program leading to the Bachelor of Architecture degree. Students study the third year of the program in Rome. The university is known for its Notre Dame School of Architecture, focusing on traditional and classical architecture and urban planning. Undergraduate traditions include Painting The Rock, Dance Marathon, Primal Scream, and Armadillo Day.'}

In [21]:
agent("can you summarize these facts in two short sentences")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


```json
{
    "action": "Final Answer",
    "action_input": "The University of Notre Dame was established in 1842 by Father Edward Sorin. It is known for its School of Architecture and various undergraduate traditions."
}
```

> Finished chain.


{'input': 'can you summarize these facts in two short sentences',
 'chat_history': [HumanMessage(content='when was the college of engineering in the University of Notre Dame established?'),
  AIMessage(content='The specific establishment date of the College of Engineering at the University of Notre Dame is not available.'),
  HumanMessage(content='what is 2 * 7?'),
  AIMessage(content='The result of 2 * 7 is 14.'),
  HumanMessage(content='can you tell me some facts about the University of Notre Dame?'),
  AIMessage(content='The University of Notre Dame was established in 1842 by Father Edward Sorin. The School of Architecture was established in 1899, offering a five-year undergraduate program leading to the Bachelor of Architecture degree. Students study the third year of the program in Rome. The university is known for its Notre Dame School of Architecture, focusing on traditional and classical architecture and urban planning. Undergraduate traditions include Painting The Rock, Dance 

<br>

Looks great! We're also able to ask questions that refer to previous interactions in the conversation and the agent is able to refer to the conversation history to as a source of information.

That's all for this example of building a retrieval augmented conversational agent with OpenAI and Pinecone (the OP stack) and LangChain.

Once finished, we delete the Pinecone index to save resources:

In [22]:
pc.delete_index(index_name)

---